In [101]:
import os
import sys
from tqdm import tqdm
import numpy as np
import torch
import random
import pandas as pd
import torch.nn.functional as F
import matplotlib.pyplot as plt
from torch.optim import SGD, Adam
from torch.utils.data import Dataset, DataLoader
from scipy.linalg import eig
from sklearn.metrics import f1_score, precision_score, recall_score
from sklearn.manifold import TSNE
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, confusion_matrix
from sklearn.metrics.pairwise import rbf_kernel, linear_kernel

In [102]:
torch.cuda.is_available()

True

In [103]:
base_url = os.getcwd()

# Intended Task

In [104]:
TASKS = ["Cross-Position", "Cross-Person", "Cross-Device"]
task = None
while True:
    print("1. Cross-Position\n2. Cross-Person\n3. Cross-Device")
    user_input = input("Please enter your task: ")
    if user_input == "1":
        task = "Cross-Position"
    elif user_input == "2":
        task = "Cross-Person"
    elif user_input == "3":
        task = "Cross-Device"
    if task in TASKS:
        break
    else:
        print("Invalid task. Please choose a valid task.")
print(f"Chosen task: {task}")

1. Cross-Position
2. Cross-Person
3. Cross-Device
Please enter your task: 
Invalid task. Please choose a valid task.
1. Cross-Position
2. Cross-Person
3. Cross-Device
Please enter your task: 1
Chosen task: Cross-Position


# Dataset Selection

In [105]:
DATASETS = ["OPP", "DSADS", "PAMAP", "WISDM", "HHAR"]
dataset = None
if task == 'Cross-Position':
    while True:
        print("1. OPP\n2. DSADS\n3. PAMAP\n")
        user_input = input("Please enter your dataset: ")
        if user_input == "1":
            dataset = "OPP"
        elif user_input == "2":
            dataset = "DSADS"
        elif user_input == "3":
            dataset = "PAMAP"
        if dataset in DATASETS:
            break
        else:
            print("Invalid dataset. Please choose a valid dataset.")
elif task == 'Cross-Person':
    while True:
        print("1. OPP\n2. DSADS\n3. PAMAP\n4. WISDM\n")
        user_input = input("Please enter your dataset: ")
        if user_input == "1":
            dataset = "OPP"
        elif user_input == "2":
            dataset = "DSADS"
        elif user_input == "3":
            dataset = "PAMAP"
        elif user_input == "4":
            dataset = "WISDM"
        if dataset in DATASETS:
            break
        else:
            print("Invalid dataset. Please choose a valid dataset.")
print(f"Chosen dataset: {dataset}")

1. OPP
2. DSADS
3. PAMAP

Please enter your dataset: 2
Chosen dataset: DSADS


# Reproduciblitiy (Must Change Before Each Experiment)

In [106]:
os.environ["CUBLAS_WORKSPACE_CONFIG"]=":4096:8"

def set_seed(seed=42, loader=None):
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    torch.use_deterministic_algorithms(True, warn_only=True)
    torch.manual_seed(seed)
    np.random.seed(seed)
    random.seed(seed)
    try:
        loader.sampler.generator.manual_seed(seed)
    except AttributeError:
        pass

seeds = [0, 1, 42, 123, 1234]
seed = seeds[0] # Change this
set_seed(seed=seed)

In [107]:
# All of these have to taken input from shell script
overlap = 0.0

if task == "Cross-Position":
    if dataset == "DSADS":
        win_size = 5*25
        domains = {0:"TORSO", 1:"RA", 2:"LA", 3:"RL", 4:"LL"}
        sources = [0, 1, 2, 3]
        nsources = len(sources)
        target = 4
        activities = [ "standing", "lying-back", "ascending", "walking-parking-lot", "treadmill-running", "stepper-exercise", "cross-trainer-exercise", "rowing", "jumping",  "playing-basketball"]
        activity_num = len(activities)
        item = ["train","valid","test"]
        person_list = ["User1","User2","User3","User4", "User5","User6","User7","User8"]
        position_array = ["TORSO","RA","LA","RL","LL"]
    elif dataset == "OPP":
        win_size = 5*30
        domains = {0:"BACK", 1:"RUA", 2:"RLA", 3:"LUA", 4:"LLA"}
        sources = [1, 2, 3, 4]
        nsources = len(sources)
        target = 0
        activities = ['Sitting','Standing','Lying','Walking']
        activity_num = len(activities)
        item = ["train","valid","test"]
        person_list = ["U1","U2","U3","U4"]
        position_array = ["BACK", "RUA", "RLA", "LUA","LLA"]
    elif dataset == "PAMAP":
        domains = {0:"Wrist", 1:"Chest", 2:"Ankle"}
        sources = [0, 2]
        nsources = len(sources)
        target = 1
        activities = ['ascending', 'lying', 'sitting', 'standing', 'walking', 'descending', 'vacuum', 'ironing', 'running', 'cycling', 'rope_jumping']
        activity_num = len(activities)
        item = ["train","valid","test"]
        person_list = ["U1","U2","U3","U4","U5","U6","U7","U8"]
        position_array = ['Wrist', 'Chest', 'Ankle']
        
elif task == "Cross-Person":
    if dataset == "DSADS":
        domains = {0:"User1", 1:"User2", 2:"User3", 3:"User4", 4:"User5", 5:"User6", 6:"User7", 7:"User8"}
        sources = [4, 5, 6, 7]
        target = [0, 1, 2, 3]
        nsources = len(sources)
        position_list = ["TORSO", "RA", "LA", "RL", "LL"]
        activities = [ "standing", "lying-back", "ascending", "walking-parking-lot", "treadmill-running", "stepper-exercise", "cross-trainer-exercise", "rowing", "jumping",  "playing-basketball"]
        activity_num = len(activities)
        item = ["train","valid","test"]
        person_array = ["User1","User2","User3","User4","User5","User6","User7","User8"]
    elif dataset == "OPP":
        domains = {0:"U1", 1:"U2", 2:"U3", 3:"U4"}
        sources = [0, 1]
        target = [2, 3]
        nsources = len(sources)
        position_list = ["BACK", "RUA", "RLA", "LUA","LLA"]
        activities = ['Sitting','Standing','Lying','Walking']
        activity_num = len(activities)
        item = ["train","valid","test"]
        person_array = ["U1","U2","U3","U4"]
    elif dataset == "PAMAP":
        domains = {0:'U1', 1:'U2', 2:'U3', 3:'U4', 4:'U5', 5:'U6', 6:'U7', 7:'U8'}
        sources = [0, 1, 2, 3]
        nsources = len(sources)
        target = [4, 5, 6, 7]
        activities = ['ascending', 'lying', 'sitting', 'standing', 'walking', 'descending', 'vacuum', 'ironing', 'running', 'cycling', 'rope_jumping']
        activity_num = len(activities)
        item = ["train","valid","test"]
        position_list = ['Wrist', 'Chest', 'Ankle']
        person_array = ["U1","U2","U3","U4","U5","U6","U7","U8"]
elif task == "Cross-Device":
    pass

print(activity_num)

10


In [108]:
step_size=int(win_size*(1-overlap))
gpu_id= 0
DEVICE = torch.device('cuda:'+str(gpu_id) if torch.cuda.is_available() else 'cpu')

In [109]:
if dataset == "DSADS":
    AXIS = 81
    FROM = 0
    TO = FROM+81
    START = 82
    END = 83
    N_FEATURES = 81
elif dataset == "OPP":
    AXIS = 81
    FROM = 0
    TO = FROM+81
    START = 82
    END = 83
    N_FEATURES = 81
elif dataset == "PAMAP":
    AXIS = 81
    FROM = 0
    TO = FROM+81
    START = 82
    END = 83
    N_FEATURES = 81

In [110]:
if task == "Cross-Position":
    if dataset == "DSADS":
        folder_name = str(activity_num)+ "_Activity"+"_Window_"+str(win_size)+ "_Overlap_"+str(overlap)
        dataset_path = base_url + "/Dataset_Preprocessed/DSADS/Data-Files-27-Features-Acc-Gyro-Mag/"+folder_name+"/"
        save_path = base_url + "/Output/DSADS-Cross-Position-Heterogeneity/"+folder_name+"/Target-Position-" + position_array[target] + '/seed' + str(seed) + "/"
        main_folder = "DSADS-AllPerson-DifferentPosition"
    elif dataset == "OPP":
        folder_name = str(activity_num)+ "_Activity"+"_Window_"+str(win_size)+ "_Overlap_"+str(overlap)
        dataset_path = base_url + "/Dataset_Preprocessed/OPPORTUNITY/Data-Files-27-Features-Acc-Gyro-Mag/"+folder_name+"/"
        save_path = base_url + "/Output/OPP-Cross-Position-Heterogeneity/"+folder_name+"/Target-Position-" + position_array[target] + '/seed' + str(seed) + "/"
        main_folder = "OPP-AllPerson-DifferentPosition"
    elif dataset == "PAMAP":
        folder_name = str(activity_num)+ "_Activity"+"_Window_"+str(win_size)+ "_Overlap_"+str(overlap)
        dataset_path = base_url + "/Dataset_Preprocessed/PAMAP/Data-Files-27-Features-Acc-Gyro-Mag/"+folder_name+"/"
        save_path = base_url + "/Output/PAMAP-Cross-Position-Heterogeneity/"+folder_name+"/Target-Position-" + position_array[target] + '/seed' + str(seed) + "/"
        main_folder = "PAMAP-AllPerson-DifferentPosition"
        
elif task == "Cross-Person":
    target_string = "_".join([domains[i] for i in target])
    if dataset == "DSADS":
        folder_name = str(activity_num)+ "_Activity"+"_Window_"+str(win_size)+ "_Overlap_"+str(overlap)
        dataset_path = base_url + "/Dataset_Preprocessed/DSADS/Data-Files-27-Features-Acc-Gyro-Mag/"+folder_name+"/"
        save_path = base_url + "/Output/DSADS-Cross-Person-Heterogeneity/"+folder_name+"/Target-" + target_string + '/seed' + str(seed) + "/"
        main_folder = "DSADS-SamePosition-DifferentPerson"
    elif dataset == "OPP":
        folder_name = str(activity_num)+ "_Activity"+"_Window_"+str(win_size)+ "_Overlap_"+str(overlap)
        dataset_path = base_url + "/Dataset_Preprocessed/OPPORTUNITY/Data-Files-27-Features-Acc-Gyro-Mag/"+folder_name+"/"
        save_path = base_url + "/Output/OPP-Cross-Person-Heterogeneity/"+folder_name+"/Target-" + target_string + '/seed' + str(seed) + "/"
        main_folder = "OPP-SamePosition-DifferentPerson"
    elif dataset == "PAMAP":
        folder_name = str(activity_num)+ "_Activity"+"_Window_"+str(win_size)+ "_Overlap_"+str(overlap)
        dataset_path = base_url + "/Dataset_Preprocessed/PAMAP/Data-Files-27-Features-Acc-Gyro-Mag/"+folder_name+"/"
        save_path = base_url + "/Output/PAMAP-Cross-Person-Heterogeneity/"+folder_name+"/Target-" + target_string + '/seed' + str(seed) + "/"
        main_folder = "PAMAP-AllPerson-DifferentPerson"

if not os.path.exists(save_path):
    os.makedirs(save_path)

# Hyperparamters

In [111]:
# All of these have to taken input from shell script
# the target and source domain position
LEARNING_RATE = 0.01
WEIGHT_DECAY = 0.0005
MOMENTUM = 0.9
# BATCH_SIZE = [200, 56]
BATCH_SIZE = 32
EPOCHS = 20
THETA1 = 0.33
THETA2 = 0.34
THETA3 = 0.33
PLOT_ITER = 10

In [112]:
s_train = None
t_train = None

s_gt_train = None
t_gt_train = None

#===================================================#

s_valid = None
t_valid = None

s_gt_valid = None
t_gt_valid = None

#===================================================#

s_test = None
t_test = None

s_gt_test = None
t_gt_test = None

In [113]:
def load_data(df, col_start, col_end, gt_start, gt_end):
    # Extract feature columns and convert to NumPy array
    data = df.iloc[:, col_start:col_end].values
    # Extract ground truth columns and convert to NumPy array
    gt = df.iloc[:, gt_start:gt_end].values.ravel()
    return data, gt

class data_loader(Dataset):
    def __init__(self, samples, labels, transform=None):
        self.samples = samples
        self.labels = labels
        self.transform = transform

    def __getitem__(self, index):
        return self.samples[index], self.labels[index]

    def __len__(self):
        return len(self.samples)

    
def load_train_valid_test(train_x, train_y, valid_x, valid_y, test_x, test_y, batch_size=64, drop_last=True):

    train_set = data_loader(train_x, train_y)
    valid_set = data_loader(valid_x, valid_y)
    test_set = data_loader(test_x, test_y)
    
    train_loader = DataLoader(train_set, batch_size=batch_size, shuffle=True, drop_last=drop_last)
    valid_loader = DataLoader(valid_set, batch_size=batch_size, shuffle=True, drop_last=drop_last)
    test_loader = DataLoader(test_set, batch_size=batch_size, shuffle=True, drop_last=drop_last)
    return train_loader, valid_loader, test_loader

In [114]:
# sources can be [0, 1, 3, 4], and target = 2
if task == 'Cross-Position':
    print('Source')
    df_aggregated_train = pd.DataFrame()
    df_aggregated_valid = pd.DataFrame()
    df_aggregated_test = pd.DataFrame()
    for i in range(len(sources)):
        # Create an empty DataFrame to hold the aggregated data for this source position
        # Combine data from all users for the current source position
        for person in person_list:
            for split_index in range(0, 3):
                file_name = person + "_" + position_array[sources[i]] + '_' + item[split_index]
                print(f"Processing: {file_name}")
                df = pd.read_csv(dataset_path + file_name + '.csv', sep=",")
                
                # Aggregate data into the corresponding split DataFrame
                if split_index == 0:
                    df_aggregated_train = pd.concat([df_aggregated_train, df], ignore_index=True)
                elif split_index == 1:
                    df_aggregated_valid = pd.concat([df_aggregated_valid, df], ignore_index=True)
                elif split_index == 2:
                    df_aggregated_test = pd.concat([df_aggregated_test, df], ignore_index=True)
        
    # Pass the aggregated data to calculate_window
    s_train, s_gt_train = load_data(df_aggregated_train, FROM, TO, START, END)
    s_valid, s_gt_valid = load_data(df_aggregated_valid, FROM, TO, START, END)
    s_test, s_gt_test = load_data(df_aggregated_test, FROM, TO, START, END)

    # For the target position, similarly aggregate data across all users
    print('Target')
    df_aggregated_train_target = pd.DataFrame()
    df_aggregated_valid_target = pd.DataFrame()
    df_aggregated_test_target = pd.DataFrame()
    for person in person_list:
        for split_index in range(0, 3):
            file_name = person + "_" + position_array[target] + '_' + item[split_index]
            print(f"Processing: {file_name}")
            df = pd.read_csv(dataset_path + file_name + '.csv', sep=",")
            
            if split_index == 0:
                df_aggregated_train_target = pd.concat([df_aggregated_train_target, df], ignore_index=True)
            elif split_index == 1:
                df_aggregated_valid_target = pd.concat([df_aggregated_valid_target, df], ignore_index=True)
            elif split_index == 2:
                df_aggregated_test_target = pd.concat([df_aggregated_test_target, df], ignore_index=True)

    t_train, t_gt_train = load_data(df_aggregated_train_target, FROM, TO, START, END)
    t_valid, t_gt_valid = load_data(df_aggregated_valid_target, FROM, TO, START, END)
    t_test, t_gt_test = load_data(df_aggregated_test_target, FROM, TO, START, END)
    
elif task == 'Cross-Person':
    # Create an empty DataFrame to hold the aggregated data for this source position
    df_aggregated_train = pd.DataFrame()
    df_aggregated_valid = pd.DataFrame()
    df_aggregated_test = pd.DataFrame()
    for i in range(len(sources)):
        for position in position_list:
            for split_index in range(0, 3):
                file_name = person_array[sources[i]] + "_" + position + '_' + item[split_index]
                print(f"Processing: {file_name}")
                df = pd.read_csv(dataset_path + file_name + '.csv', sep=",")
                
                # Aggregate data into the corresponding split DataFrame
                if split_index == 0:
                    df_aggregated_train = pd.concat([df_aggregated_train, df], ignore_index=True)
                elif split_index == 1:
                    df_aggregated_valid = pd.concat([df_aggregated_valid, df], ignore_index=True)
                elif split_index == 2:
                    df_aggregated_test = pd.concat([df_aggregated_test, df], ignore_index=True)
        
    # Pass the aggregated data to calculate_window
    s_train, s_gt_train = load_data(df_aggregated_train, s_train, s_gt_train, FROM, TO, START, END)
    s_valid, s_gt_valid = load_data(df_aggregated_valid, s_valid, s_gt_valid, FROM, TO, START, END)
    s_test, s_gt_test = load_data(df_aggregated_test, s_test, s_gt_test, FROM, TO, START, END)

    # For the target position, similarly aggregate data across all users
    df_aggregated_train_target = pd.DataFrame()
    df_aggregated_valid_target = pd.DataFrame()
    df_aggregated_test_target = pd.DataFrame()
    for i in range(len(target)):
        for position in position_list:
            for split_index in range(0, 3):
                file_name = person_array[target[i]] + "_" + position + '_' + item[split_index]
                print(f"Processing: {file_name}")
                df = pd.read_csv(dataset_path + file_name + '.csv', sep=",")

                if split_index == 0:
                    df_aggregated_train_target = pd.concat([df_aggregated_train_target, df], ignore_index=True)
                elif split_index == 1:
                    df_aggregated_valid_target = pd.concat([df_aggregated_valid_target, df], ignore_index=True)
                elif split_index == 2:
                    df_aggregated_test_target = pd.concat([df_aggregated_test_target, df], ignore_index=True)

    t_train, t_gt_train = load_data(df_aggregated_train_target, t_train, t_gt_train, FROM, TO, START, END)
    t_valid, t_gt_valid = load_data(df_aggregated_valid_target, t_valid, t_gt_valid, FROM, TO, START, END)
    t_test, t_gt_test = load_data(df_aggregated_test_target, t_test, t_gt_test, FROM, TO, START, END)

elif task == "Cross-Device":
    pass

Source
Processing: User1_TORSO_train
Processing: User1_TORSO_valid
Processing: User1_TORSO_test
Processing: User2_TORSO_train
Processing: User2_TORSO_valid
Processing: User2_TORSO_test
Processing: User3_TORSO_train
Processing: User3_TORSO_valid
Processing: User3_TORSO_test
Processing: User4_TORSO_train
Processing: User4_TORSO_valid
Processing: User4_TORSO_test
Processing: User5_TORSO_train
Processing: User5_TORSO_valid
Processing: User5_TORSO_test
Processing: User6_TORSO_train
Processing: User6_TORSO_valid
Processing: User6_TORSO_test
Processing: User7_TORSO_train
Processing: User7_TORSO_valid
Processing: User7_TORSO_test
Processing: User8_TORSO_train
Processing: User8_TORSO_valid
Processing: User8_TORSO_test
Processing: User1_RA_train
Processing: User1_RA_valid
Processing: User1_RA_test
Processing: User2_RA_train
Processing: User2_RA_valid
Processing: User2_RA_test
Processing: User3_RA_train
Processing: User3_RA_valid
Processing: User3_RA_test
Processing: User4_RA_train
Processing: Us

In [115]:
print(type(s_train))
print(len(s_train))
print(s_train.shape)

<class 'numpy.ndarray'>
21888
(21888, 81)


In [116]:
print(type(s_gt_train))
print(len(s_gt_train))
print(s_gt_train.shape)

<class 'numpy.ndarray'>
21888
(21888,)


In [117]:
s_train.shape

(21888, 81)

In [118]:
def kernel(ker, X1, X2, gamma):
    K = None
    if not ker or ker == 'primal':
        K = X1
    elif ker == 'linear':
        if X2 is not None:
            K = linear_kernel(np.asarray(X1).T, np.asarray(X2).T)
        else:
            K = linear_kernel(np.asarray(X1).T)
    elif ker == 'rbf':
        if X2 is not None:
            K = rbf_kernel(np.asarray(X1).T, np.asarray(X2).T, gamma)
        else:
            K = rbf_kernel(np.asarray(X1).T, None, gamma)
    return K


class TCA:
    def __init__(self, kernel_type='primal', dim=30, lamb=1, gamma=1):
        '''
        Init func
        :param kernel_type: kernel, values: 'primal' | 'linear' | 'rbf'
        :param dim: dimension after transfer
        :param lamb: lambda value in equation
        :param gamma: kernel bandwidth for rbf kernel
        '''
        self.kernel_type = kernel_type
        self.dim = dim
        self.lamb = lamb
        self.gamma = gamma

    def fit(self, Xs, Xt):
        '''
        Transform Xs and Xt
        :param Xs: ns * n_feature, source feature
        :param Xt: nt * n_feature, target feature
        :return: Xs_new and Xt_new after TCA
        '''
        X = np.hstack((Xs.T, Xt.T))
        X /= np.linalg.norm(X, axis=0)
        m, n = X.shape
        ns, nt = len(Xs), len(Xt)
        e = np.vstack((1 / ns * np.ones((ns, 1)), -1 / nt * np.ones((nt, 1))))
        M = e * e.T
        M = M / np.linalg.norm(M, 'fro')
        H = np.eye(n) - 1 / n * np.ones((n, n))
        K = kernel(self.kernel_type, X, None, gamma=self.gamma)
        n_eye = m if self.kernel_type == 'primal' else n
        a, b = K @ M @ K.T + self.lamb * np.eye(n_eye), K @ H @ K.T
        w, V = eig(a, b)
        ind = np.argsort(w)
        A = V[:, ind[:self.dim]]
        Z = A.T @ K
        Z /= np.linalg.norm(Z, axis=0)

        Xs_new, Xt_new = Z[:, :ns].T, Z[:, ns:].T
        return Xs_new, Xt_new

    def fit_predict(self, Xs, Ys, Xt, Yt):
        '''
        Transform Xs and Xt, then make predictions on target using 1NN
        :param Xs: ns * n_feature, source feature
        :param Ys: ns * 1, source label
        :param Xt: nt * n_feature, target feature
        :param Yt: nt * 1, target label
        :return: Accuracy and predicted_labels on the target domain
        '''
        Xs_new, Xt_new = self.fit(Xs, Xt)
        clf = KNeighborsClassifier(n_neighbors=1)
        clf.fit(Xs_new, Ys.ravel())
        y_pred = clf.predict(Xt_new)
        acc = accuracy_score(Yt, y_pred)

        return acc, y_pred

In [119]:
# # Assuming TCA class is already defined as per your provided code
# # from your_tca_module import TCA

# def stl(X_src, y_src, X_tar, y_tar, dim):
#     """
#     Stratified Transfer Learning for Cross-domain Activity Recognition.
    
#     Parameters:
#     ----------
#     X_src : numpy.ndarray
#         Source domain feature matrix of shape (n_samples, n_dim)
#     y_src : numpy.ndarray
#         Source domain label vector of shape (n_samples,)
#     X_tar : numpy.ndarray
#         Target domain feature matrix of shape (n_samples, n_dim)
#     y_tar : numpy.ndarray
#         Target domain label vector of shape (n_samples,)
#     dim : int
#         Dimension after STL
    
#     Returns:
#     -------
#     acc_stl : float
#         Accuracy of the algorithm
#     """
#     label_set = np.unique(y_src)
#     num_tree = 30

#     # Define and train classifiers
#     rf = RandomForestClassifier(n_estimators=num_tree)
#     svm = SVC(C=100, probability=True)
#     knn = KNeighborsClassifier(n_neighbors=5)

#     rf.fit(X_src, y_src)
#     svm.fit(X_src, y_src)
#     knn.fit(X_src, y_src)

#     # Predict on target domain
#     predict_label_rf = rf.predict(X_tar)
#     predict_label_svm = svm.predict(X_tar)
#     predict_label_knn = knn.predict(X_tar)

#     # Voting
#     label_voting = np.where(predict_label_rf == predict_label_svm, predict_label_rf, -1)
#     label_voting = np.where(predict_label_rf == predict_label_knn, predict_label_rf, label_voting)
#     label_voting = np.where(predict_label_svm == predict_label_knn, predict_label_svm, label_voting)

#     r_index = (label_voting == -1)
#     X_tar_residual = X_tar[r_index]
#     y_tar_residual = y_tar[r_index]

#     c_index = (label_voting != -1)
#     X_tar_candidate = X_tar[c_index]
#     y_tar_candidate = y_tar[c_index]
#     label_candidate = label_voting[c_index]

#     conmat1 = confusion_matrix(label_candidate, y_tar_candidate)

#     # In-class transfer
#     X_src_new = []
#     y_src_new = []
#     X_tar_candidate_new = []
#     y_tar_candidate_new = []
#     X_tar_candidate_old = []
#     y_tar_candidate_old = []

#     # Initialize TCA
#     tca = TCA(kernel_type='linear', dim=dim, lamb=1, gamma=1)

#     for class_index in label_set:
#         # Source data in that class
#         src_index_i = (y_src == class_index)
#         X_src_i = X_src[src_index_i]
#         y_src_i = y_src[src_index_i]

#         # Candidate data in that class
#         tar_index_i = (label_candidate == class_index)
#         X_tar_candidate_i = X_tar_candidate[tar_index_i]
#         y_tar_candidate_i = y_tar_candidate[tar_index_i]

#         X_tar_candidate_old.extend(X_tar_candidate_i)
#         y_tar_candidate_old.extend(y_tar_candidate_i)

#         # Apply TCA
#         X_src_new_i, X_tar_candidate_new_i = tca.fit(X_src_i, X_tar_candidate_i)

#         # Merge
#         X_src_new.extend(X_src_new_i)
#         y_src_new.extend(y_src_i)
#         X_tar_candidate_new.extend(X_tar_candidate_new_i)
#         y_tar_candidate_new.extend(y_tar_candidate_i)

#     X_src_new = np.array(X_src_new)
#     y_src_new = np.array(y_src_new)
#     X_tar_candidate_new = np.array(X_tar_candidate_new)
#     y_tar_candidate_new = np.array(y_tar_candidate_new)
#     X_tar_candidate_old = np.array(X_tar_candidate_old)
#     y_tar_candidate_old = np.array(y_tar_candidate_old)

#     # Train model on new candidate
#     rf_new = RandomForestClassifier(n_estimators=num_tree)
#     rf_new.fit(X_src_new, y_src_new)
#     predicted_label_candidate = rf_new.predict(X_tar_candidate_new)
#     acc_candidate = accuracy_score(y_tar_candidate_new, predicted_label_candidate)

#     conmat2 = confusion_matrix(predicted_label_candidate, y_tar_candidate_new)

#     # Train model on old candidate and predict on residual
#     rf_old = RandomForestClassifier(n_estimators=num_tree)
#     rf_old.fit(X_tar_candidate_old, predicted_label_candidate)
#     predicted_label_residual = rf_old.predict(X_tar_residual)
#     acc_residual = accuracy_score(y_tar_residual, predicted_label_residual)

#     # Calculate accuracy
#     acc_stl = np.mean(np.concatenate([predicted_label_candidate, predicted_label_residual]) ==
#                       np.concatenate([y_tar_candidate_new, y_tar_residual]))

#     return acc_stl

In [120]:
# drop_last = True
# S_train, S_valid, S_test = load_train_valid_test(s_train, s_gt_train, s_valid, s_gt_valid, s_test, s_gt_test, BATCH_SIZE, drop_last)
# T_train, T_valid, T_test = load_train_valid_test(t_train, t_gt_train, t_valid, t_gt_valid, t_test, t_gt_test, BATCH_SIZE, drop_last)

In [121]:
import numpy as np
from sklearn import svm
from sklearn.metrics import accuracy_score
from sklearn.preprocessing import label_binarize

def jdsvm(x_train, y_train, x_test, y_test, c=None, g=None, kernel_type=None):
    predict_label = []
    accuracy = 0
    prob_estimates = []
    
    c_default = 1
    g_default = 1 / x_train.shape[1]
    kernel_type_default = 'rbf'

    c_para = c_default
    g_para = g_default
    kernel_para = kernel_type_default

    if c is None and g is None and kernel_type is None:
        pass
    elif c is not None and g is None and kernel_type is None:
        c_para = c
    elif c is not None and g is not None and kernel_type is None:
        c_para = c
        g_para = g
    elif c is not None and g is not None and kernel_type is not None:
        c_para = c
        g_para = g
        kernel_para = kernel_type
    else:
        raise ValueError('++++++Fatal error! Please check the input!')

    model = svm.SVC(C=c_para, gamma=g_para, kernel=kernel_para, probability=True)
    model.fit(x_train, y_train)
    predict_label = model.predict(x_test)
    accuracy = accuracy_score(y_test, predict_label)
    prob_estimates = model.predict_proba(x_test)

    return predict_label, accuracy, prob_estimates

In [122]:
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier

def jdrf(x_train, y_train, x_test, y_test, n_trees=10):
    """
    jdrf: packaged RandomForestClassifier from sklearn, just for easier use.
    
    input:
    ------ x_train,...,y_test  :   training and testing sets.        [required]
    ------ n_trees             :   number of trees, default is 10.    [not required]

    output:
    ------ predict_label       :   predicted label vector for test case.
    ------ accuracy            :   accuracy
    """
    predict_label = []
    accuracy = 0

    tree_para = n_trees

    model = RandomForestClassifier(n_estimators=tree_para)
    model.fit(x_train, y_train)
    predict_labels = model.predict(x_test)
    accuracy = np.mean(predict_labels == y_test)
    predict_label = predict_labels

    return predict_label, accuracy



def jdknn(x_train, y_train, x_test, y_test, k=None, distance='euclidean', rule='nearest'):
    predict_label = []
    accuracy = 0
    k_default = 5
    distance_default = 'euclidean'
    rule_default = 'nearest'

    if k is None:
        k_para = k_default
    else:
        k_para = k

    # Note: distance and rule parameters are not directly used in sklearn's KNeighborsClassifier
    knn_model = KNeighborsClassifier(n_neighbors=k_para)
    knn_model.fit(x_train, y_train)

    predict_label = knn_model.predict(x_test)
    accuracy = np.sum(predict_label == y_test) / len(y_test)

    return predict_label, accuracy

In [123]:
import numpy as np
from sklearn.metrics import confusion_matrix

def STL(X_src, y_src, X_tar, y_tar, dim):
    # Stratified Transfer Learning for Cross-domain Activity Recognition.
    # Jindong Wang, Yiqiang Chen, Lisha hu, Xiaohui Peng, and Philip S. Yu.
    # 2018 IEEE International Conference on Pervasive Computing and Communications (PerCom).

    # Inputs: 
    # X_src : Source domain feature matrix, n_sample * n_dim
    # y_src : Source domain label vector, n_sample * 1
    # X_tar : Target domain feature matrix, n_sample * n_dim
    # y_tar : Target domain label vector, n_sample * 1
    # dim   : Dimension after STL, an integer
    # Output:
    # acc_stl: accuracy of the algorithm

    label_set = np.unique(y_src)
    num_tree = 30

    # Voting
    # RF
    predict_label_rf, accuracy_rf = jdrf(X_src, y_src, X_tar, y_tar, 30)
    # SVM
    predict_label_svm, accuracy_svm, _ = jdsvm(X_src, y_src, X_tar, y_tar, 100)
    # KNN
    predict_label_knn, accuracy_knn = jdknn(X_src, y_src, X_tar, y_tar, 5)

    # Voting
    label_voting = []
    for i in range(len(y_tar)):
        if predict_label_rf[i] == predict_label_svm[i]:
            label_voting.append(predict_label_rf[i])
        elif predict_label_rf[i] == predict_label_knn[i]:
            label_voting.append(predict_label_rf[i])
        elif predict_label_svm[i] == predict_label_knn[i]:
            label_voting.append(predict_label_svm[i])
        else:
            label_voting.append(-1)

    label_voting = np.array(label_voting)
    
    # Get residual
    r_index = label_voting == -1
    X_tar_residual = X_tar[r_index]
    y_tar_residual = y_tar[r_index]
    
    # Get candidate
    c_index = label_voting != -1
    X_tar_candidate = X_tar[c_index]
    y_tar_candidate = y_tar[c_index]
    label_candidate = label_voting[c_index]
    conmat1 = confusion_matrix(label_candidate, y_tar_candidate)

    # In-class transfer
    # New candidates and source domain
    X_tar_candidate_new = []
    y_tar_candidate_new = []
    X_src_new = []
    y_src_new = []
    X_tar_candidate_old = []
    y_tar_candidate_old = []
    
    # Initialize TCA
    tca = TCA(kernel_type='linear', dim=dim, lamb=1, gamma=1)

    for class_index in label_set:
        # Source data in that class
        src_index_i = (y_src == class_index)
        X_src_i = X_src[src_index_i]
        y_src_i = y_src[src_index_i]
        
        # Candidate data in that class
        tar_index_i = (label_candidate == class_index)
        X_tar_candidate_i = X_tar_candidate[tar_index_i]
        y_tar_candidate_i = y_tar_candidate[tar_index_i]
        
        X_tar_candidate_old.append(X_tar_candidate_i)
        y_tar_candidate_old.append(y_tar_candidate_i)
        
        # TCA
        X_src_new_i, X_tar_candidate_new_i = tca.fit(X_src_i, X_tar_candidate_i)
        
        # Merge
        X_src_new.append(X_src_new_i)
        y_src_new.append(y_src_i)
        X_tar_candidate_new.append(X_tar_candidate_new_i)
        y_tar_candidate_new.append(y_tar_candidate_i)

    # Convert lists to arrays
    X_src_new = np.vstack(X_src_new)
    y_src_new = np.concatenate(y_src_new)
    X_tar_candidate_new = np.vstack(X_tar_candidate_new)
    y_tar_candidate_new = np.concatenate(y_tar_candidate_new)
    X_tar_candidate_old = np.vstack(X_tar_candidate_old)
    y_tar_candidate_old = np.concatenate(y_tar_candidate_old)

    # Train model on new candidate
    predicted_label_candidate, acc_candidate = jdrf(X_src_new, y_src_new, X_tar_candidate_new, y_tar_candidate_new, num_tree)
    conmat2 = confusion_matrix(predicted_label_candidate, y_tar_candidate_new)

    # Train model on old candidate and predict on residual
    predicted_label_residual, acc_residual = jdrf(X_tar_candidate_old, predicted_label_candidate, X_tar_residual, y_tar_residual, num_tree)

    # Calculate accuracy
    acc_stl = np.mean(np.concatenate([predicted_label_candidate, predicted_label_residual]) == np.concatenate([y_tar_candidate_new, y_tar_residual]))

    return acc_stl

In [124]:
def main():
    file_name = 'DSADS'  # Options: 'OPP', 'DSADS', 'PAMAP'
    dim = 30  # Dimension after STL, should be less than 81

    for i in range(1, 2):  # Indexing in Python starts from 0
        for j in range(2, 3):
            if i == j:
                continue
            # src_domain, tar_domain = load_data(file_name, i, j)
            src_domain, tar_domain = s_train, t_test

            n1 = src_domain.shape[0]
            n2 = tar_domain.shape[0]
            seq1 = np.random.permutation(n1)
            seq2 = np.random.permutation(n2)
            r_source_domain = src_domain[seq1]
            r_target_domain = tar_domain[seq2]
            
            x_src = r_source_domain
            y_src = s_gt_train
            x_tar = r_target_domain
            y_tar = t_gt_test

            scaler = StandardScaler()
            x_src = scaler.fit_transform(x_src)
            x_tar = scaler.transform(x_tar)

            acc_stl = STL(x_src, y_src, x_tar, y_tar, dim) * 100
            print(f'Acc: {acc_stl:.2f}')

if __name__ == "__main__":
    main()

Acc: 5.44
